### Lab Assignment: Python Warmup and Logfile Analytics

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: January 23, 2026

---

This lab consists of two parts:

- Part 1 is the Python warmup
- Part 2 is the logfile analytics in PySpark

Answer the questions in this assignment, showing all code and solutions.

**Total points: 20**

---

### Part 1: Python Warmup

1) (1 PT) Rename this notebook to JupyterTutorial_[your_initials], where you will enter your initials in place of [your_initials].

2) (1 PT) In the cell below, enter a list of data science topics you find interesting.  Use the markdown style (you will need to change the style from the Code style).

- Time Series Analysis
- Biostatistics
- Genomics
- Political Science

3) (1 PT) In the cell below, enter the following Python list:  

some_vals = [1, 6, 10, 44]  

You will use the Code style, run the cell, and print the list.

In [61]:
some_vals = [1,6,10,44]
some_vals

[1, 6, 10, 44]

4) (1 PT) Use a list comprehension to return a filtered list containing only the values greater than 6.  
Call this list *some_vals_filtered* and print it.

In [62]:
[val for val in some_vals if val > 6]

[10, 44]

Next, a small pandas dataframe is constructed.

In [63]:
import pandas as pd

df = pd.DataFrame({'first_name': ['Andy','Crystal'],
                   'domain_facebook' : [1,1],
                   'domain_foursquare' : [0,0],
                   'age' : [20, 32]})
df

,first_name,domain_facebook,domain_foursquare,age
0,Andy,1,0,20
1,Crystal,1,0,32


5) (1 PT) In the cell below, write a list comprehension that returns the fields names in the dataframe `df` containing the string *domain*.  Run the cell to verify the correct result.

In [64]:
[col for col in df.columns if 'domain' in col]

['domain_facebook', 'domain_foursquare']

6) (1 PT) Use the list comprehension from (5) to index into `df` and show the data for columns containing *domain*

In [65]:
domain_cols = [col for col in df.columns if 'domain' in col]

df[domain_cols]

,domain_facebook,domain_foursquare
0,1,0
1,1,0


7) (1 PT) In the cell below, print the *domain_facebook* column

In [66]:
df['domain_foursquare']

0    0
1    0
Name: domain_foursquare, dtype: int64

8) (1 PT) In the cell below, print the row with index 1.

In [67]:
df.loc[[1]]

,first_name,domain_facebook,domain_foursquare,age
1,Crystal,1,0,32


9) (1 PT) Next, you will cube the *age* column of `df` and assign the result to a new column called *agecube*.

Specifically, call the `apply` method with a `lambda function` inside to cube the *age* column.  
Print the dataframe.

In [68]:
df = df.assign(agecube = lambda x: (x['age'] ** 3))
df

,first_name,domain_facebook,domain_foursquare,age,agecube
0,Andy,1,0,20,8000
1,Crystal,1,0,32,32768


10) (1 PT) Given the list of strings below, form one string, placing semicolons between each word.  It should look like this:  

`'the;quick;brown;fox'`

Print the resulting string.

In [69]:
some_list = ['the','quick','brown','fox']

In [70]:
some_str = ";".join(some_list)
some_str

'the;quick;brown;fox'

---

### Part 2: Logfile Analytics

Import modules for Spark Session and regex 

Note: regexes can be used to search strings for patterns. Here is a [reference](https://realpython.com/regex-python/?utm_source=chatgpt.com).

In [71]:
from pyspark.sql import SparkSession
import re

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [72]:
import socket
print(socket.gethostname())
print(socket.gethostbyname(socket.gethostname()))

dhcp017104095235.corp.apple.com
17.104.95.235


11) (1 PT) Read in the logfile.txt data

In [73]:
log_rdd = sc.textFile("logfile.txt")

12) (1 PT) Count the number of rows of data

In [74]:
row_count = log_rdd.count()
row_count

100000

13) (2 PTS) Show the first five lines containing WARN.  
    Write code to count and print the total number of lines containing WARN.

In [75]:
warn_rdd = log_rdd.filter(lambda x: "WARN" in x)

first_five_warn = warn_rdd.take(5)
for line in first_five_warn:
    print(line)

warn_count = warn_rdd.count()
warn_count

2026-01-01T08:00:09.764000 [WARN] scheduler: heartbeat ip=92.174.240.101 latency_ms=475 trace=nw6i1m40mwvc seq=19
2026-01-01T08:00:12.828000 [WARN] api: partition reassigned ip=84.161.222.95 latency_ms=4691 trace=yht26vhtm0eh seq=26
2026-01-01T08:00:14.718000 [WARN] auth: job completed ip=230.79.10.241 latency_ms=3732 trace=tedo8mtq5qu9 seq=29
2026-01-01T08:00:20.259000 [WARN] api: shuffle completed ip=212.54.74.42 latency_ms=4063 trace=ej2u544h7nwc seq=40
2026-01-01T08:00:28.523000 [WARN] auth: cache miss ip=242.94.66.209 latency_ms=753 trace=feqvh0k4dwun seq=57


14823

14) (2 PTS) Write a word count program to count the number of each of these log levels:  

- WARN
- INFO
- DEBUG
- ERROR

In [76]:
# Check to see if filtering would be beneficial
info_count = log_rdd.filter(lambda x: "INFO" in x).count()
print(info_count)

debug_count = log_rdd.filter(lambda x: "DEBUG" in x).count()
print(debug_count)

error_count = log_rdd.filter(lambda x: "ERROR" in x).count()
print(error_count)

sum([warn_count, info_count, debug_count, error_count])

70083
5143
9951


100000

In [77]:
words_rdd = log_rdd.flatMap(lambda x: x.split(" "))

log_levels_rdd = words_rdd.filter(lambda x: x in ["[WARN]", "[INFO]", "[DEBUG]", "[ERROR]"])

pairs_rdd = log_levels_rdd.map(lambda x: (x, 1))

counts_rdd = pairs_rdd.reduceByKey(lambda a, b: a + b)

counts_rdd.collect()

[('[WARN]', 14823), ('[INFO]', 70083), ('[ERROR]', 9951), ('[DEBUG]', 5143)]

15. (2 PTS) Return the three log lines with the highest latency. This is reported in the log as `latency_ms`.

Note: There may be more than three lines tied for highest latency, in which case, just show three records.

In [78]:
latency_pairs = log_rdd.map(lambda x: (int((re.search(r"latency_ms=(\d+)", x)).group(1)), x))

top_3 = latency_pairs.top(3)

[line[1] for line in top_3]

['2026-01-01T19:23:31.069000 [WARN] metrics: stream started ip=20.185.253.192 latency_ms=5000 trace=wo5u4hfibjja seq=86365',
 '2026-01-01T18:32:23.860000 [INFO] metrics: shuffle completed ip=6.154.28.129 latency_ms=5000 trace=uvhstvgu9ar7 seq=79893',
 '2026-01-01T17:35:14.572000 [INFO] ingest: schema validated ip=254.68.147.151 latency_ms=5000 trace=c1ojcdws1fg5 seq=72635']

16. (2 PTS) Compute the average latency for each service. Ignore log entries without a latency_ms.

In [79]:
latency_lines = log_rdd.filter(lambda x: re.search(r"latency_ms=(\d+)", x))
latency_lines.count()

service_pairs = log_rdd.map(lambda x: ((re.search(r"\]\s+(\w+):", x).group(1)), int(re.search(r"latency_ms=(\d+)", x).group(1))))

service_value_pairs = service_pairs.map(lambda x: (x[0], (x[1], 1)))

service_totals = service_value_pairs.reduceByKey(lambda a, b: ((a[0] + b[0]), (a[1] + b[1])))

service_averages = service_totals.map(lambda x: (x[0], (x[1][0] / x[1][1])))

service_averages.collect()

[('api', 2503.894314115308),
 ('metrics', 2485.1066211495713),
 ('auth', 2506.9450742455692),
 ('storage', 2505.294059730883),
 ('gateway', 2492.284061901723),
 ('scheduler', 2483.6426985888697),
 ('spark', 2508.2136396381975),
 ('ingest', 2501.4536561898653)]